# Inspect saved video embeddings
Read existing R(2+1)D embeddings and optional detection metadata. No encoder or detector is run,
and no output files are written. Similarity checks below are exploratory, not held-out evaluation.


In [ ]:
import sys
from pathlib import Path

import numpy as np
from IPython.display import HTML, display

# Support kernels started from the repository root or the notebooks folder.
ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "src").is_dir() and (path / "main.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start this notebook inside the repository")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.inspection import load_recordings, load_embedding, detection_rows, table_html

OUTPUT_DIR = ROOT / "outputs/visual_baseline"
RAW_DIR = ROOT / "data/raw/reassemble"
recordings = load_recordings(OUTPUT_DIR)
print("Available recordings:", *recordings, sep="\n")


In [ ]:
# Change this ID to inspect another completed recording.
VIDEO_ID = next(iter(recordings))
recording = recordings[VIDEO_ID]
manifest = recording["manifest"]
clips = recording["clips"]
display(manifest)
print("Clips:", len(clips))


In [ ]:
# Change this index to inspect another clip.
CLIP_INDEX = 0
clip = clips[CLIP_INDEX]
embedding = load_embedding(recording, clip)
print("Clip:", clip["clip_id"], "Label:", clip["ground_truth"])
print("Embedding path:", recording["directory"] / clip["embedding_path"])
print("Shape:", embedding.shape, "Dtype:", embedding.dtype)
print("Norm:", np.linalg.norm(embedding))
print("First ten values:", embedding[:10])
display(manifest["config"])


## Validate all saved embeddings
Each path is resolved from its own recording directory.


In [ ]:
rows = []
for video_id, item in recordings.items():
    matrix = np.stack([load_embedding(item, c) for c in item["clips"]])
    norms = np.linalg.norm(matrix, axis=1)
    rows.append({"video_id": video_id, "clips": len(matrix), "dimensions": matrix.shape[1],
                 "min_norm": norms.min(), "max_norm": norms.max()})
display(HTML(table_html(rows)))


## Similar clips within the selected recording
The query itself is excluded. Neighbors can overlap in time; this is not a classifier metric.


In [ ]:
matrix = np.stack([load_embedding(recording, c) for c in clips])
normalized = matrix / np.linalg.norm(matrix, axis=1, keepdims=True)
similarities = normalized @ normalized[CLIP_INDEX]
ranked = [int(i) for i in np.argsort(-similarities) if i != CLIP_INDEX][:5]
display(HTML(table_html([{
    "clip_id": clips[i]["clip_id"], "ground_truth": clips[i]["ground_truth"],
    "segment_id": clips[i]["segment_id"], "cosine_similarity": float(similarities[i])
} for i in ranked])))


### Detections from earlier runs
`object` identifies the piece named in the activity annotation.
`detections` stores sampled frames as `{"timestamp": relative_video_seconds, "objects": [...]}`.
Each object has `label`, `confidence` and `bbox` (`[x1, y1, x2, y2]` in pixels).
The current batch writes an empty list. Use this view to inspect detections saved by earlier runs.


In [ ]:
detection_columns = ["recording_id", "clip_id", "timestamp", "label", "confidence", "bbox"]
detections = detection_rows(clips)
print("YOLO enabled for this run:", manifest["config"]["use_yolo"])
print("Stored sampled frames:", sum(len(c.get("detections", []) or []) for c in clips))
print("Stored object detections:", len(detections))
display(HTML(table_html(detections[:20], detection_columns)))


In [ ]:
# Ground-truth objects and detected objects are separate sources of information.
print("Selected clip:", clip["clip_id"])
print("Annotated object:", clip.get("object"))
selected_detections = detection_rows([clip])
display(HTML(table_html(selected_detections, detection_columns)))


## Prototype Classifier top-k results
Run `main.py prepare-experiment` and `main.py evaluate-experiment` first.
This section only reads saved predictions. Select an experiment, partition and support seed below.
Validation is the default. Inspect test results only after settings are fixed.


In [ ]:
import json

EXPERIMENT_DIR = ROOT / "outputs/experiments/prototype_5shot"
PARTITION = "validation"
SUPPORT_SEED = 0
result_path = EXPERIMENT_DIR / PARTITION / f"seed_{SUPPORT_SEED}.json"
if not result_path.exists():
    print("No saved predictions yet:", result_path)
    print("Prepare an experiment and evaluate validation first.")
else:
    result = json.loads(result_path.read_text())
    display(result["metrics"])
    rows = [{"clip_id": prediction["clip_id"],
             "ground_truth": prediction["ground_truth"],
             "predicted_activity": prediction["predicted_activity"],
             "top_k": ", ".join(f"{candidate['action']} ({candidate['score']:.4f})"
                                for candidate in prediction["top_k"])}
            for prediction in result["predictions"][:20]]
    display(HTML(table_html(rows)))
